# Oracle Cache-Then-Memory Mode

This notebook demonstrates `retrieval_mode="cache_then_memory"`.

What you should see:

1. Run 1 calls Tavily and writes rows as `cache_plus_memory`.
2. The notebook waits for the short cache TTL to expire.
3. Run 2 falls through the expired cache tier and retrieves from durable Oracle memory.
4. The final cell deletes this notebook's demo rows so reruns remain clear.

## Setup

Run this cell first. It loads `.env`, connects to Oracle, prepares the demo table, and imports shared notebook helpers.

In [1]:
from pathlib import Path
import sys

helper_path = None
for root in [Path.cwd(), *Path.cwd().parents]:
    candidate = root / "examples" / "oracle" / "demo_helpers.py"
    if candidate.exists():
        helper_path = candidate
        break

if helper_path is None:
    candidate = Path.cwd() / "demo_helpers.py"
    if candidate.exists():
        helper_path = candidate

if helper_path is None:
    raise RuntimeError("Could not find examples/oracle/demo_helpers.py. Start Jupyter from the repository root or examples/oracle.")

sys.path.insert(0, str(helper_path.parent))
from demo_helpers import *

print("Using helper:", helper_path)

Loaded environment from /Users/anishraj/Desktop/Projects/Pod4_demo/tavily-python/.env
Keeping proxy variables because TAVILY_USE_ENV_PROXY=1
Connected to Oracle. Table: TAVILY_DOCUMENTS
Table exists. Schema is ready.
Demo helpers are ready. Scores are ranking signals, not probabilities.
Using helper: /Users/anishraj/Desktop/Projects/Pod4_demo/tavily-python/examples/oracle/demo_helpers.py


## Choose the demo query

Edit only `MEMORY_QUERY` below when you want to try your own cache-then-memory demo query.

In [2]:
MEMORY_QUERY = "Oracle Tavily cache then memory retrieval lifecycle"

## Start from a clean query-specific state

In [3]:
delete_rows_for_query(MEMORY_QUERY)

Deleted 0 old demo rows for this query.


0

## Run once, let cache expire, then run again

The TTL is intentionally `1` second so you can see memory behavior quickly.

In [4]:
memory_client = make_client(
    "cache_then_memory",
    persistence_depth="cache_plus_memory",
    cache_ttl_seconds=1,
    cache_score_threshold=-1.0,
    memory_score_threshold=-1.0,
    memory_max_results=5,
)

first_results = memory_client.search(
    MEMORY_QUERY,
    max_results=3,
    max_local=3,
    max_foreign=3,
    save_foreign=True,
    **TAVILY_SEARCH_OPTIONS,
)
show_results("Run 1: no Oracle memory yet, Tavily fallback", first_results)
show_persisted_rows(MEMORY_QUERY, "Rows after Run 1")

print("Waiting for the 1-second cache TTL to expire...")
time.sleep(2)

second_results = memory_client.search(
    MEMORY_QUERY,
    max_results=3,
    max_local=3,
    max_foreign=3,
    save_foreign=True,
    **TAVILY_SEARCH_OPTIONS,
)
show_results("Run 2: cache expired, Oracle memory still works", second_results)
show_persisted_rows(MEMORY_QUERY, "Rows after Run 2")

### Run 1: no Oracle memory yet, Tavily fallback
`total=3` `origins={'local': 3}`

| rank | origin | score | preview |
| --- | --- | --- | --- |
| 1 | local | 0.0496 | In this proof-of-concept demonstration at Oracle CloudWorld, we showed the integration of Oracle Database 23ai with... |
| 2 | local | 0.0496 | In this proof-of-concept demonstration at Oracle CloudWorld, we showed the integration of Oracle Database 23ai with... |
| 3 | local | 0.0106 | There are several components to an effective service model including the necessary product support infrastructure,... |

### Rows after Run 1

No rows.

Waiting for the 1-second cache TTL to expire...


### Run 2: cache expired, Oracle memory still works
`total=3` `origins={'local': 3}`

| rank | origin | score | preview |
| --- | --- | --- | --- |
| 1 | local | 0.0496 | In this proof-of-concept demonstration at Oracle CloudWorld, we showed the integration of Oracle Database 23ai with... |
| 2 | local | 0.0496 | In this proof-of-concept demonstration at Oracle CloudWorld, we showed the integration of Oracle Database 23ai with... |
| 3 | local | 0.0106 | There are several components to an effective service model including the necessary product support infrastructure,... |

### Rows after Run 2

No rows.

## Cleanup

Run this at the end. It deletes this notebook's demo rows so the next full run starts with Tavily again.

In [5]:
delete_rows_for_query(MEMORY_QUERY)
print("Cleaned cache-then-memory demo rows. Re-run from the top to see Run 1 call Tavily again.")

Deleted 0 old demo rows for this query.
Cleaned cache-then-memory demo rows. Re-run from the top to see Run 1 call Tavily again.
